In [4]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path.cwd() / "testfiles_" / "data"

factor_returns_df = pd.read_csv(DATA_DIR / "test11_2_factor_returns.csv", header=0)
stock_returns_df = pd.read_csv(DATA_DIR / "test11_2_stock_returns.csv", header=0)
beta_df = pd.read_csv(DATA_DIR / "test11_2_beta.csv", header=0)
weights_df = pd.read_csv(DATA_DIR / "test11_2_weights.csv", header=0)

factor_ret = factor_returns_df.select_dtypes(include=[np.number]).to_numpy(dtype=float)
stock_ret = stock_returns_df.select_dtypes(include=[np.number]).to_numpy(dtype=float)
beta = beta_df.select_dtypes(include=[np.number]).to_numpy(dtype=float)
w_init = weights_df["W"].to_numpy(dtype=float)

T = stock_ret.shape[0]
N_stocks = stock_ret.shape[1]
N_factors = factor_ret.shape[1]

w_dynamic = np.zeros((T, N_stocks))
port_ret = np.zeros(T)
w_curr = w_init.copy()

for t in range(T):
    w_dynamic[t] = w_curr
    w_after_ret = w_curr * (1.0 + stock_ret[t])
    sum_w = np.sum(w_after_ret)
    port_ret[t] = sum_w - 1.0
    w_curr = w_after_ret / sum_w

factor_contrib_ts = np.zeros((T, N_factors))
for t in range(T):
    for f in range(N_factors):
        factor_contrib_ts[t, f] = np.sum(w_dynamic[t] * beta[:, f]) * factor_ret[t, f]

# Total factor and portfolio returns
cumulative_factor_ret = np.exp(np.sum(np.log(1.0 + factor_ret), axis=0)) - 1.0
port_cumulative_ret = np.exp(np.sum(np.log(1.0 + port_ret))) - 1.0

# Carino weights
k_factor = np.log(1.0 + port_cumulative_ret) / port_cumulative_ret
carino_weights = np.log(1.0 + port_ret) / (port_ret * k_factor)

# Return attribution
factor_attribution = np.zeros(N_factors)
for f in range(N_factors):
    factor_attribution[f] = np.sum(factor_contrib_ts[:, f] * carino_weights)

weighted_factor_contrib = factor_contrib_ts
design_matrix = np.vstack([np.ones(T), port_ret]).T
coefficients = np.linalg.lstsq(design_matrix, weighted_factor_contrib, rcond=None)[0]
slopes = coefficients[1]

# Vol attribution
port_std = np.std(port_ret, ddof=1)
vol_attribution = slopes * port_std

# Alpha pieces: use the same residual for TotalReturn and Return Attribution
alpha_attr = port_cumulative_ret - np.sum(factor_attribution)
alpha_total = alpha_attr
alpha_vol = port_std - np.sum(vol_attribution)

factor_names = factor_returns_df.select_dtypes(include=[np.number]).columns.tolist()

# Header
print(f"Value,{','.join(factor_names)},Alpha,Portfolio")

# TotalReturn row
print("TotalReturn", end="")
for val in cumulative_factor_ret:
    print(f",{val:.17f}", end="")
print(f",{alpha_total:.17f},{port_cumulative_ret:.17f}")

# Return Attribution row
print("Return Attribution", end="")
for val in factor_attribution:
    print(f",{val:.17f}", end="")
print(f",{alpha_attr:.17f},{port_cumulative_ret:.17f}")

# Vol Attribution row
print("Vol Attribution", end="")
for val in vol_attribution:
    print(f",{val:.17f}", end="")
print(f",{alpha_vol:.17f},{port_std:.17f}")


Value,F1,F2,F3,Alpha,Portfolio
TotalReturn,0.07993902853336143,0.02671083578339073,-0.01631718718235609,0.04755891337183536,0.10532667972875132
Return Attribution,0.06517280614688026,0.00017358190287438,-0.00757862169283868,0.04755891337183536,0.10532667972875132
Vol Attribution,0.00501279608465526,-0.00000919256617865,0.00178472450241195,0.00544518653022976,0.01223351455111831
